In [ ]:
# Cell 1 - install indepencies
# Direct imports: torch, transformers, peft, trl, datasets, huggingface_hub
# Installed but used internally: bitsandbytes (quantization), accelerate (device management)
%%capture
!pip install transformers bitsandbytes accelerate peft trl datasets huggingface_hub

In [ ]:
# Cell 2 - Authenticate with Hugging Face
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

if hf_token:
  login(token=hf_token)


In [ ]:
# Cell 3 - Confirm GPU
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
Device: Tesla T4
VRAM: 15.6 GB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs("/content/drive/MyDrive/healthcare-llm/checkpoints", exist_ok=True)
print("Drive mounted. Checkpoints will save here.")

Mounted at /content/drive
Drive mounted. Checkpoints will save here.


In [ ]:
# Cell - 4 Load Cleaned Dataset
from datasets import load_dataset

dataset = load_dataset("nicholas-ugbala-hf/chatdoctor-cleaned-10k")
train_dataset = dataset['train'].select(range(5000))
eval_dataset = dataset['eval']

print(f"Train: {len(train_dataset)}")
print(f"Eval: {len(eval_dataset)}")

README.md:   0%|          | 0.00/1.48k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/11.7M [00:00<?, ?B/s]

data/eval-00000-of-00001.parquet:   0%|          | 0.00/1.30M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9000 [00:00<?, ? examples/s]

Generating eval split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Train: 5000
Eval: 1000


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.add_special_tokens({'pad_token': '<pad>'})
model.resize_token_embeddings(len(tokenizer))
tokenizer.padding_side = "right"
tokenizer.model_max_length = 512

print("Model loaded.")

Loading model...


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model loaded.


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 9,175,040 || all params: 3,221,927,936 || trainable%: 0.2848


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Filter out any sequences over 512 tokens
def filter_long_sequences(sample):
    tokens = tokenizer(sample['text'], truncation=False)['input_ids']
    return len(tokens) <= 512

train_dataset = train_dataset.filter(filter_long_sequences)
eval_dataset  = eval_dataset.filter(filter_long_sequences)

print(f"Train after length filter: {len(train_dataset)}")
print(f"Eval after length filter:  {len(eval_dataset)}")

Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (544 > 512). Running this sequence through the model will result in indexing errors


Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Train after length filter: 4937
Eval after length filter:  990


In [9]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="/content/drive/MyDrive/healthcare-llm/checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=4,       # back to 4
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,       # back to 4, effective batch = 16
    warmup_steps=50,
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    gradient_checkpointing=True,         # add this
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()
print("Training complete.")

Adding EOS to train dataset:   0%|          | 0/4937 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4937 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/990 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/990 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128256}.


Starting training...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,2.569781,2.557117,2.556207,410828.000000,0.477880
200,2.524198,2.510934,2.529786,821552.000000,0.483833


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


KeyboardInterrupt: 